In [1]:
import torch
import torch.nn.functional as F

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from model import QASpanDetector
model = torch.load('models/yolo_qa_v3.pth', map_location=torch.device('cpu'))
# model = QASpanDetector()

In [3]:
from transformers.data.metrics.squad_metrics import compute_f1

In [4]:
import pickle

with open('data/debug_preprocessed2.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [5]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

In [7]:
MODEL_MAX_LENGTH = 512

In [8]:
import numpy as np

def create_labels(encodings):
    encodings['answer_length'] = np.array(encodings['end_positions'])\
        - np.array(encodings['start_positions']) + 1

    labels = np.zeros((len(encodings['input_ids']), MODEL_MAX_LENGTH, 
        2)) # num_example * seq_length * 2

    for example_idx, start in enumerate(encodings['start_positions']):
        if start < MODEL_MAX_LENGTH: # if the answer is not truncated
            labels[example_idx, start, 0] = 1
            labels[example_idx, start, 1] = encodings['answer_length'][example_idx]

    encodings['labels'] = labels

    return encodings

In [24]:
device = "cuda" if torch.cuda.is_available() else 'cpu'

model.eval()

print("############Train############")
for batch_idx, batch in enumerate(train_loader):
    if batch_idx < 4:
        continue
    batch = create_labels(batch)
    sentence_length = batch['input_ids'].size(1)
    batch_size = batch['input_ids'].size(0)

    answer_start = batch['start_positions'].reshape(batch_size, 1).to(device)        
    attention_mask = batch['attention_mask']
    attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

    reg_target = batch['labels'][:, :, 1]
    obj_target = batch['labels'][:, :, 0]
    obj_target = torch.FloatTensor(obj_target).to(device)
    reg_target = torch.FloatTensor(reg_target).to(device)

    outputs = model(batch['input_ids'], batch['attention_mask'])

    print(outputs)
    break


############Train############
TokenClassifierOutput(loss=None, logits=tensor([[[-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
         [-4.6857,  1.1995],
  

In [19]:
torch.max(outputs['logits'][0, :, 0].sigmoid())

tensor(0.0091, grad_fn=<MaxBackward1>)

In [12]:
obj_pred = outputs['logits'][:, :, 0]

In [13]:
obj_pred.shape

torch.Size([1, 451])

In [14]:
obj_target = torch.zeros(obj_pred.shape)
obj_target[:, 0] = 1
obj_target

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0.,

In [15]:
import torch.nn.functional as F

print(F.binary_cross_entropy_with_logits(obj_pred, obj_target, reduction="sum"))

tensor(8.8278, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [16]:
obj_pred

tensor([[-4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857, -4.6857,
         -4.6857, -4.6857, -

In [17]:
F.binary_cross_entropy_with_logits(torch.Tensor([-4.6857]), torch.Tensor([0]))

tensor(0.0092)